In [17]:
import sys
import json
import os
import subprocess
from dotenv import load_dotenv
import openai

print("Environment loaded!")

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print(" No API key was found - please double check your .env configuration file.")
elif not api_key.startswith("sk-proj-"):
    print(" An API key was found, but it doesn't match the modern 'sk-proj-' template syntax.")
elif api_key.strip() != api_key:
    print(" Key found, but it contains hidden empty leading/trailing spaces.")
else:
    print(" OpenAI API Key verified and initialized successfully!")

Environment loaded!
 OpenAI API Key verified and initialized successfully!


In [21]:
scraper_code = '''
import sys
from playwright.sync_api import sync_playwright

def fetch_page(url):
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url, wait_until="networkidle")
        text = page.locator("body").inner_text()
        browser.close()
        return text

if __name__ == "__main__":
    url = sys.argv[1]
    print(fetch_page(url))
'''

with open("scraper.py", "w", encoding="utf-8") as f:
    f.write(scraper_code)

print(" scraper.py created!")

 scraper.py created!


In [23]:
def fetch_website_playwright_sync(url):
    print(f" Launching background browser to visit: {url}...")
    result = subprocess.run(
        [sys.executable, "scraper.py", url],
        capture_output=True, text=True, timeout=60,
        encoding="utf-8"
    )
    if result.returncode != 0:
        print(" Scraper script error:", result.stderr)
        return ""
    return result.stdout


def build_system_prompt(extraction_goal=None):
    goal = extraction_goal or (
        "Identify the main repeating content on the page (for example products, "
        "quotes, articles, job listings, or table rows) and extract each one."
    )
    return f"""You are a precise data extraction bot. You will receive raw text scraped from a web page.

Your task: {goal}

Rules:
- Use only information present in the text. Never invent data.
- Choose clear, consistent field names and use the same fields for every item.
- If the page has no repeating items, return an empty "items" list and put the key information in "summary".

Respond with a valid JSON object in exactly this layout:
{{
    "page_title": "Title or topic of the page",
    "page_type": "e.g. quotes list, product listing, article, homepage",
    "summary": "1-2 sentence summary of the page",
    "items": [
        {{"field_name": "value"}}
    ]
}}"""


def extract_structured_data_sync(url, api_key, extraction_goal=None, max_chars=30000):
    website_text = fetch_website_playwright_sync(url)

    print(f" Scraped {len(website_text)} characters")

    if not website_text or len(website_text.strip()) < 50:
        print(" Error: Target web page returned zero or very little content.")
        return None

    if len(website_text) > max_chars:
        print(f" Text too long, trimming to first {max_chars} characters")
        website_text = website_text[:max_chars]

    messages = [
        {"role": "system", "content": build_system_prompt(extraction_goal)},
        {"role": "user", "content": f"Extract the data from this scraped page text:\n\n{website_text}"}
    ]

    print(" Processing through OpenAI...")
    client = openai.OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
url_to_test = "https://www.seedprod.com/best-blogging-platforms/"
goal = None  # leave as None to let the model decide, or e.g. "Extract every quote with its author"

try:
    data = extract_structured_data_sync(url_to_test, api_key, extraction_goal=goal)

    if data:
        print("\n🚀 ===== EXTRACTION SUCCESSFUL ===== 🚀")
        print(f"Title:   {data.get('page_title')}")
        print(f"Type:    {data.get('page_type')}")
        print(f"Summary: {data.get('summary')}\n")

        items = data.get("items", [])
        print(f"Found {len(items)} items. Showing first 5:\n")
        for item in items[:5]:
            for key, value in item.items():
                print(f"  {key}: {value}")
            print("-" * 55)
    else:
        print("⚠️ No data could be extracted.")

except Exception as err:
    print(f"❌ Execution crashed with message: {err}")

 Launching background browser to visit: https://www.blogger.com/dashboard/reading...
 Scraped 968 characters
 Processing through OpenAI...

🚀 ===== EXTRACTION SUCCESSFUL ===== 🚀
Title:   Sign in to Blogger
Type:    login page
Summary: This page provides a sign-in interface for Blogger, including language options and support links.

Found 0 items. Showing first 5:

